# Read mapping and gene-biotype content: ONT subcellular fractions (HCT116) and APEX-seq (HEK293T)

As `pbody_RNA_Spang.ReadMapping.ipynb`, for the two data sets of this project:
- **ONT**: all libraries of the FACS-sorted subcellular fractions (Free, Org2, ER, TIS, Golgi, PEX, Endo, TJ; siCtrl and siCPSF5; three batches), the HCT116 Total libraries (wt, siCtrl, siCPSF6), the input-amount control and the unlabelled End/TJ barcodes (`nanoflowz`; labels as in `ONT_organelles_vs_Total.ipynb`);
- **APEX-seq** (Fazal et al. 2019, GSE116008): labelled targets of 9 constructs, unlabelled controls (the reference) and nuclear / cytosolic fractionation (bulk RNA-seq workflow, run `APEX_seq_Fazal2019`; labels as in `APEX_seq_vs_Control.ipynb`).

Sections, for each data set:
1. reads along the processing pipeline;
2. composition of the reads assigned to genes by gene class (nuclear protein-coding, mitochondrially encoded, lncRNA, small RNAs, rRNA, other);
3. composition by every gene biotype of the annotation (GENCODE v42 basic `gene_type`; chrM genes by their biotype: Mt_rRNA, Mt_tRNA, protein-coding);
4. enrichment of each biotype in the fractions against the reference: the share of the biotype relative to nuclear protein-coding reads, $p = n_b / (n_b + n_{pc})$ (which removes the varying shares of other classes), and the odds ratio fraction vs reference, as for the Ty elements in the P-body notebook.

Finally the two data sets side by side for the references and the matching fractions.

**Counting.** ONT: reads after UMI deduplication, featureCounts as in `nanoflowz` (one read per molecule). APEX-seq: read pairs, featureCounts on unique alignments, reverse-stranded (`Snakefile-quantification-faster`, `FeatureCounts_standard`). The read shares are shares of reads (or read pairs), not length-corrected: for short reads, long RNAs give more reads per molecule, so the APEX-seq shares of long RNA classes (e.g. lncRNAs such as MALAT1) are larger than their share of molecules. Both use poly(A)+ RNA: ONT reads are selected for a poly(A) tail, APEX-seq libraries are TruSeq poly(A)+.

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import csv
import glob

import numpy as np
import pandas as pd

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR"),
           "lab_group_dir": os.getenv("LAB_GROUP_DIR")}
ANALYSIS = "ONT_organelles_and_APEX_ReadMapping"
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)
for d in (FIGS, TABLES):
    os.makedirs(d, exist_ok=True)

# ONT: nanoflowz runs and the library table of ONT_organelles_vs_Total.ipynb
RUNS = {"Total": os.path.join(subdirs["lab_group_dir"], "APA_AI_data", "wf_runs", "v2p0_ONT_HCT116_v1", "results"),
        "ER_TIS_Golgi_PEX": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_ER_TIS_PEX_Golgi_Aleksei_v1", "results_alignment_with_SpikeIns"),
        "End_TJ": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_End_TJ_Aleksei_v1", "results_alignment_with_SpikeIns"),
        "input_control": os.path.join(subdirs["wf_runs_dir"], "v2p0_ONT_FACSorganelles_Aleksei_v1", "results_alignment_with_SpikeIns")}
ONT_TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", "ONT_organelles_vs_Total")
# APEX-seq: bulk workflow run and the inputs of WF/scripts/APEX_fracSanity_pipeline.py
APEX_WF = os.path.join(subdirs["wf_runs_dir"], "APEX_seq_Fazal2019", "output")
APEX_SU = os.path.join(subdirs["wf_runs_dir"], "Sanity_utils_APEX_seq_Fazal2019")
APEX_READY = os.path.exists(os.path.join(APEX_SU, "inputs", "done"))
GTF = os.path.join(subdirs["lab_group_dir"], "Genomes", "homo_sapiens", "hg38_v42", "gencode.v42.basic.annotation.gtf")

def figure_caption(path, formats=("png", "pdf")):
    # file name of a saved figure, shown with the plot; links relative to the HTML report (make_html_reports.py)
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, name, formats=("png", "pdf")):
    for fmt in formats:
        fig.savefig(os.path.join(FIGS, f"{name}.{fmt}"), bbox_inches="tight", dpi=300)
    figure_caption(os.path.join(FIGS, name), formats)

def save_tsv(df, name, index=False):
    df.to_csv(os.path.join(TABLES, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)

# notation of the two QC notebooks: colour = fraction / construct, marker = condition (ONT) or kind of library (APEX-seq)
ONT_FRACTION_ORDER = ["Total", "Free", "Org2", "ER", "TIS", "Golgi", "PEX", "Endo", "TJ", "End/TJ (unlabelled)", "Total 1000 ng", "Total 100 ng", "other"]
ONT_COLORS = {"Total": "black", "Free": "tab:blue", "Org2": "tab:orange", "ER": "tab:green", "TIS": "tab:red", "Golgi": "tab:purple",
              "PEX": "tab:brown", "End/TJ (unlabelled)": "grey", "Total 1000 ng": "darkslategrey", "Total 100 ng": "tab:cyan",
              "Endo": "tab:pink", "TJ": "tab:olive", "other": "lightgrey"}
ONT_MARKERS = {"siCtrl": "o", "siCPSF5": "^", "wt": "s", "siCPSF6": "D", "unknown": "X"}
APEX_FRACTION_ORDER = ["Total", "NLS", "NIK", "LMA", "NucPore", "NES", "ERM", "KDEL", "OMM", "MITO", "Nuc", "Cyto"]
APEX_COLORS = {"Total": "black", "NLS": "tab:red", "NIK": "darkred", "LMA": "tab:orange", "NucPore": "goldenrod", "NES": "tab:blue",
               "ERM": "tab:green", "KDEL": "yellowgreen", "OMM": "tab:purple", "MITO": "tab:pink", "Nuc": "tab:brown", "Cyto": "tab:cyan"}
APEX_MARKERS = {"control": "D", "target": "o", "fractionation": "s"}

# gene annotation: biotype and class of every gene
genes = []
for chunk in pd.read_csv(GTF, sep="\t", comment="#", header=None, usecols=[0, 2, 8], names=["chrom", "type", "attr"], chunksize=500_000):
    genes.append(chunk.loc[chunk["type"] == "gene"])
genes = pd.concat(genes)
genes = pd.DataFrame({"chrom": genes["chrom"].to_numpy(),
                      "gene_type": genes["attr"].str.extract(r'gene_type "([^"]+)"')[0].to_numpy(),
                      "gene_name": genes["attr"].str.extract(r'gene_name "([^"]+)"')[0].to_numpy()},
                     index=genes["attr"].str.extract(r'gene_id "([^"]+)"')[0].to_numpy())
SMALL = ["snoRNA", "snRNA", "scaRNA", "misc_RNA", "vault_RNA", "ribozyme", "sRNA", "scRNA", "Y_RNA"]
CLASSES = ["nuclear protein-coding", "mitochondrially encoded", "lncRNA", "small RNAs", "rRNA", "pseudogenes", "other"]
CLASS_COLORS = {"nuclear protein-coding": "lightgrey", "mitochondrially encoded": "crimson", "lncRNA": "darkorange", "small RNAs": "mediumseagreen",
                "rRNA": "royalblue", "pseudogenes": "sienna", "other": "black"}
gene_class = pd.Series(np.select(
    [genes["chrom"] == "chrM", genes["gene_type"] == "protein_coding", genes["gene_type"] == "lncRNA", genes["gene_type"].isin(SMALL),
     genes["gene_type"].isin(["rRNA", "rRNA_pseudogene"]), genes["gene_type"].str.contains("pseudogene")],
    ["mitochondrially encoded", "nuclear protein-coding", "lncRNA", "small RNAs", "rRNA", "pseudogenes"], default="other"), index=genes.index)
nuclear_pc = genes.index[(genes["gene_type"] == "protein_coding") & (genes["chrom"] != "chrM")]
print(f"{len(genes):,} genes;", gene_class.value_counts().to_dict(), "| APEX-seq inputs ready:", APEX_READY)

In [ ]:
def class_composition(counts):
    # % of the assigned reads per gene class, libraries x classes
    c = counts.groupby(gene_class.reindex(counts.index).fillna("other").to_numpy()).sum()
    return (100 * c / c.sum()).T.reindex(columns=CLASSES).fillna(0)

def biotype_composition(counts):
    # % of the assigned reads per gene biotype (chrM genes as Mt_rRNA, Mt_tRNA or 'protein_coding (chrM)')
    bt = genes["gene_type"].where(~((genes["chrom"] == "chrM") & (genes["gene_type"] == "protein_coding")), "protein_coding (chrM)")
    c = counts.groupby(bt.reindex(counts.index).fillna("unknown").to_numpy()).sum()
    return (100 * c / c.sum()).T

def biotype_order_colors(comp, top=12):
    # the biotypes with the largest median share (protein_coding first), the rest as 'other biotypes'
    med = comp.median().sort_values(ascending=False)
    order = ["protein_coding"] + [b for b in med.index if b != "protein_coding"][:top - 1]
    palette = sns.color_palette("tab20", len(order))
    colors = {b: ("lightgrey" if b == "protein_coding" else palette[i]) for i, b in enumerate(order)}
    return order, colors

def enrichment_vs_protein_coding(counts, ref_libs):
    # per biotype: share relative to nuclear protein-coding reads, p = n_b / (n_b + n_pc), per library, and the odds ratio
    # library vs the pooled reference libraries
    bt = genes["gene_type"].where(genes["chrom"] != "chrM", "chrM (" + genes["gene_type"] + ")")
    c = counts.groupby(bt.reindex(counts.index).fillna("unknown").to_numpy()).sum()
    pc = counts.reindex(nuclear_pc, fill_value=0).sum()
    c = c.drop(index="protein_coding", errors="ignore")
    p = c / (c + pc)
    p_ref = c[ref_libs].sum(axis=1) / (c[ref_libs].sum(axis=1) + pc[ref_libs].sum())
    odds = lambda x: x / (1 - x)
    lor = np.log2(odds(p.clip(1e-6, 1 - 1e-6)).div(odds(p_ref.clip(1e-6, 1 - 1e-6)), axis=0))
    return 100 * p.T, lor.T

def library_axis(ax, labels, fractions, colors, markers=None, marker_of=None):
    ax.set_yticks(np.arange(len(labels)), labels, fontsize=6.5)
    for t, f in zip(ax.get_yticklabels(), fractions):
        t.set_color(colors.get(f, "black"))
    ax.set_ylim(len(labels) - 0.5, -0.5)

def stacked_barh(ax, comp, order, colors, labels, fractions, palette, xlabel):
    left = np.zeros(len(comp))
    for b in order:
        ax.barh(np.arange(len(comp)), comp[b], left=left, color=colors[b], edgecolor="black", lw=0.2, height=0.85, label=b)
        left += comp[b].to_numpy()
    rest = 100 - left
    if rest.max() > 0.01:     # biotypes not shown individually
        ax.barh(np.arange(len(comp)), rest, left=left, color="white", edgecolor="black", lw=0.2, height=0.85, label="other biotypes")
    library_axis(ax, labels, fractions, palette)
    ax.set(xlabel=xlabel, xlim=(0, 100))

## ONT subcellular fractions and Totals (HCT116)

### Samples and reads along the pipeline

Library table and read tracking (`QC/master_read_tracking_stats.tsv` of each `nanoflowz` run) as in `ONT_organelles_vs_Total.ipynb`; all labelled libraries, siCtrl and siCPSF5, and all Totals. Mapped reads (unique and multi-mapping) at each step: basecalled, mapped (minimap2), with a poly(A) tail, before and after UMI deduplication, and the deduplicated reads assigned to genes (featureCounts).

In [ ]:
libs = pd.read_csv(os.path.join(ONT_TABLES, "libraries.tsv"), sep="\t").set_index("library", drop=False).rename_axis(None)
libs = libs.loc[libs["use"] != "QC only"]
qc = pd.read_csv(os.path.join(ONT_TABLES, "QC.read_tracking_and_gene_assignments.tsv"), sep="\t").set_index("library").reindex(libs.index)

def gene_count_file(source, sample_id):
    return os.path.join(RUNS[source], "gene_assignments", f"{sample_id}.featureCounts.txt")

ont_counts = pd.DataFrame({l: pd.read_csv(gene_count_file(r["source"], r["sample_id"]), sep="\t", comment="#", index_col=0).iloc[:, -1].reindex(genes.index)
                           for l, r in libs.iterrows()}).fillna(0)
qc["assigned to genes"] = ont_counts.sum()
STEPS = ["basecalled", "mapped", "poly(A)", "before dedup", "after dedup", "assigned to genes"]
reads = qc[STEPS] / 1e6
save_tsv(libs[["batch", "fraction", "condition", "replicate", "use"]].join(qc[STEPS]).rename_axis("library").reset_index(), "ONT.read_numbers.tsv")
display(libs.groupby(["batch", "fraction", "condition"], sort=False).size().rename("libraries").to_frame().T)

labels, fracs = libs.index.tolist(), libs["fraction"].tolist()
sns.set(font_scale=0.9, style="white")
fig, ax = plt.subplots(figsize=(5.5, 0.17 * len(libs) + 1.5))
for step, color in zip(STEPS, ["lightgrey", "black", "tab:purple", "grey", "royalblue", "tab:green"]):
    ax.scatter(reads[step], np.arange(len(libs)), color=color, s=14, label=step, zorder=3)
library_axis(ax, labels, fracs, ONT_COLORS)
ax.set(xscale="log", xlabel="reads, million")
ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
for b in libs["batch"].unique()[1:]:
    ax.axhline(np.where(libs["batch"] == b)[0][0] - 0.5, color="grey", lw=0.6)
ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
sns.despine(); fig.tight_layout()
savefig(fig, "ONT.read_numbers")
plt.show()

### Gene classes and gene biotypes of the assigned reads

In [ ]:
ont_class = class_composition(ont_counts)
ont_bt = biotype_composition(ont_counts)
save_tsv(libs[["batch", "fraction", "condition", "replicate"]].join(ont_class).rename_axis("library").reset_index(), "ONT.percent_of_reads_per_gene_class.tsv")
save_tsv(libs[["batch", "fraction", "condition", "replicate"]].join(ont_bt).rename_axis("library").reset_index(), "ONT.percent_of_reads_per_biotype.tsv")
bt_order, bt_colors = biotype_order_colors(ont_bt)

sns.set(font_scale=0.85, style="white")
fig, axes = plt.subplots(1, 2, figsize=(14, 0.17 * len(libs) + 1.8), sharey=True)
stacked_barh(axes[0], ont_class, CLASSES, CLASS_COLORS, labels, fracs, ONT_COLORS, "% of the reads assigned to genes")
axes[0].legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, title="gene class")
stacked_barh(axes[1], ont_bt[bt_order], bt_order, bt_colors, labels, fracs, ONT_COLORS, "% of the reads assigned to genes")
axes[1].legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0), title="gene biotype")
for ax in axes:
    for b in libs["batch"].unique()[1:]:
        ax.axhline(np.where(libs["batch"] == b)[0][0] - 0.5, color="black", lw=0.8)
fig.tight_layout()
savefig(fig, "ONT.percent_of_reads_per_class_and_biotype")
plt.show()

grp = libs[["batch", "fraction", "condition"]]
display(ont_class.join(grp).groupby(["batch", "fraction", "condition"], sort=False).median().round(2))
display(ont_bt[bt_order].join(grp).groupby(["batch", "fraction", "condition"], sort=False).median().round(2))

### Enrichment of the biotypes in the fractions against Total

For every biotype $b$ with reads (chrM genes apart, as `chrM (<biotype>)`): $p_b = n_b / (n_b + n_{\text{nuclear pc}})$ per library, and the log2 odds ratio of each library against the pooled siCtrl Total libraries. A biotype with log2 OR > 0 is more abundant relative to nuclear mRNAs in the library than in Total. Shown: biotypes with ≥ 0.01% of the reads in the pooled Total or median ≥ 0.01% over the libraries; groups = batch, fraction, condition (median over the replicates).

In [ ]:
ref = libs.index[(libs["fraction"] == "Total") & (libs["condition"] == "siCtrl")]
ont_p, ont_lor = enrichment_vs_protein_coding(ont_counts, ref)
keep = ont_p.columns[(ont_p.loc[ref].mean() >= 0.01) | (ont_p.median() >= 0.01)]
save_tsv(libs[["batch", "fraction", "condition", "replicate"]].join(ont_lor[keep]).rename_axis("library").reset_index(), "ONT.biotype_log2_odds_ratio_vs_Total.tsv")
show = libs.loc[libs["use"].isin(["Total", "fraction", "input control"])]
g = ont_lor.loc[show.index, keep].join(show[["batch", "fraction", "condition"]]).groupby(["batch", "fraction", "condition"], sort=False).median()
g.index = [f"{b.replace(' libraries', '')}: {f} {c}" for b, f, c in g.index]
order_cols = ont_p.loc[ref, keep].mean().sort_values(ascending=False).index
g = g[order_cols]
lim = np.nanmax(np.abs(g.values[np.isfinite(g.values)]))
sns.set(font_scale=0.8, style="white")
fig, ax = plt.subplots(figsize=(0.42 * len(order_cols) + 3, 0.22 * len(g) + 2))
sns.heatmap(g, cmap="RdBu_r", center=0, vmin=-min(lim, 4), vmax=min(lim, 4), annot=True, fmt=".1f", annot_kws={"fontsize": 5.5}, ax=ax,
            cbar_kws={"label": "log2 odds ratio vs Total siCtrl\n(biotype relative to nuclear protein-coding)"})
ax.set(xlabel="", ylabel="")
for t in ax.get_yticklabels():
    t.set_color(ONT_COLORS.get(t.get_text().split(": ")[-1].rsplit(" ", 1)[0], "black"))
savefig(fig, "ONT.biotype_log2_odds_ratio_vs_Total")
plt.show()
save_tsv(g.round(3).rename_axis("group").reset_index(), "ONT.biotype_log2_odds_ratio_vs_Total.by_group.tsv")

#### Interpretation: ONT

Medians of the siCtrl libraries per batch and fraction (tables above).
- **Total (HCT116 run).** 84% of the deduplicated reads on nuclear protein-coding genes, 9.7% on mitochondrially encoded genes (5.2% mRNAs, 1.9% Mt_rRNA, 2.6% Mt_tRNA), 4.0% lncRNA, 1.4% pseudogenes, 0.5% small RNAs (snoRNA 0.16%, misc_RNA 0.34%); nuclear rRNA is practically absent (poly(A)-tailed reads only). wt, siCtrl and siCPSF6 Totals are alike.
- **The input-amount control Totals (prepared with the sorts) contain 4–7 times less mitochondrial RNA** (1.4–1.9% chrM mRNA reads, 2–5% in total) than the HCT116 Total reference. Mitochondrial content therefore differs between the RNA preparations, which matters for any comparison of the fractions with the HCT116 Total (the reference of `ONT_organelles.fracSanity.ipynb`).
- **Mitochondrial RNA dominates the ER/TIS batch and the PEX and TJ sorts.** In the ER/TIS batch, every fraction, including Free, has 35–50% mitochondrially encoded reads (and up to 97% in the libraries left out for depth in the other notebooks); in the other two batches, PEX (39%) and TJ (46%) have the most, Golgi (18%), Org2 (12–22%) and Endo (15%) less, and Free the least (2.5%). Relative to nuclear mRNAs, all three mitochondrial biotypes (mRNAs, Mt_rRNA, Mt_tRNA) are enriched together by log2 OR 2–4 (heatmap), as expected for whole mitochondria or mitochondrial fragments co-sorting with these particles, while the Free fractions of the Golgi/PEX and End/TJ batches are depleted of them (log2 OR −1.6 to −2.1). The Mt_tRNA reads are poly(A)-tailed reads assigned to the tRNA genes of the polycistronic mitochondrial transcripts.
- **Unprocessed pseudogenes follow the mitochondrial RNA** (log2 OR 2–3.4 in the mitochondria-rich fractions): most likely reads of mitochondrial transcripts assigned to nuclear mitochondrial pseudogenes (NUMTs).
- **lncRNAs keep their share relative to mRNAs** in all fractions (log2 OR within ±0.3; their share of all reads drops only because mitochondrial RNA takes a larger part), and **snoRNAs are depleted in every fraction** (log2 OR −0.2 to −1.0), consistent with their nuclear (nucleolar) localization. Other small biotypes have few reads, and their odds ratios in single libraries (e.g. misc_RNA, TEC, snRNA in ER/TIS TIS siCPSF5) are noise.
- Reads assigned to genes slightly exceed the deduplicated reads, because a read overlapping several genes counts for each of them.

## APEX-seq (HEK293T)

### Samples and reads along the pipeline

Workflow `mapping_stats` (input, mapped and uniquely mapped read pairs, as counted by STAR) and read pairs assigned to genes (featureCounts). Libraries as in `APEX_seq_vs_Control.ipynb`.

In [ ]:
if APEX_READY:
    alibs = pd.read_csv(os.path.join(APEX_SU, "inputs", "libraries.tsv"), sep="\t")
    alibs["fraction"] = alibs["fraction"].where(alibs["kind"] != "control", "Total")
    order = {f: i for i, f in enumerate(APEX_FRACTION_ORDER)}
    alibs["cells"] = np.where(alibs["kind"] == "fractionation", "", alibs["library"].str.split("-").str[1])
    alibs = alibs.assign(_k=alibs["kind"].map({"target": 0, "control": 1, "fractionation": 2}), _f=alibs["cells"].map(lambda c: order.get(c, 99)),
                         _g=alibs["fraction"].map(order)).sort_values(["_k", "_f", "_g", "library"]).drop(columns=["_k", "_f", "_g"])
    alibs = alibs.set_index("library", drop=False).rename_axis(None)
    aqc = pd.read_csv(os.path.join(APEX_SU, "inputs", "read_qc.per_library.tsv"), sep="\t", index_col=0).reindex(alibs.index)
    apex_counts = pd.read_csv(os.path.join(APEX_SU, "inputs", "gene_counts.per_library.tsv.gz"), sep="\t", index_col=0)[alibs.index]
    apex_counts = apex_counts.reindex(genes.index).fillna(0)
    areads = pd.DataFrame({"input read pairs": aqc["input_reads"], "mapped pairs": aqc["mapped_reads"],
                           "uniquely mapped pairs": aqc["uniquely_mapped_reads"], "assigned to genes": apex_counts.sum()}) / 1e6
    save_tsv(alibs[["kind", "fraction", "replicate"]].join(areads * 1e6).rename_axis("library").reset_index(), "APEX.read_numbers.tsv")
    alabels, afracs = [f"{l} ({k[0]})" for l, k in zip(alibs.index, alibs["kind"])], alibs["fraction"].tolist()
    sns.set(font_scale=0.9, style="white")
    fig, ax = plt.subplots(figsize=(5.5, 0.17 * len(alibs) + 1.5))
    for step, color in zip(areads.columns, ["lightgrey", "black", "royalblue", "tab:green"]):
        ax.scatter(areads[step], np.arange(len(alibs)), color=color, s=14, label=step, zorder=3)
    library_axis(ax, alabels, afracs, APEX_COLORS)
    ax.set(xlabel="read pairs, million")
    ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
    for k in ("control", "fractionation"):
        ax.axhline(np.where(alibs["kind"] == k)[0][0] - 0.5, color="grey", lw=0.6)
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    sns.despine(); fig.tight_layout()
    savefig(fig, "APEX.read_numbers")
    plt.show()
else:
    print("APEX-seq: inputs of WF/scripts/APEX_fracSanity_pipeline.py not ready yet (workflow APEX_seq_Fazal2019 still running)")

### Gene classes and gene biotypes of the assigned read pairs

In [ ]:
if APEX_READY:
    apex_class = class_composition(apex_counts)
    apex_bt = biotype_composition(apex_counts)
    save_tsv(alibs[["kind", "fraction", "replicate"]].join(apex_class).rename_axis("library").reset_index(), "APEX.percent_of_reads_per_gene_class.tsv")
    save_tsv(alibs[["kind", "fraction", "replicate"]].join(apex_bt).rename_axis("library").reset_index(), "APEX.percent_of_reads_per_biotype.tsv")
    abt_order, abt_colors = biotype_order_colors(apex_bt)
    sns.set(font_scale=0.85, style="white")
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.17 * len(alibs) + 1.8), sharey=True)
    stacked_barh(axes[0], apex_class, CLASSES, CLASS_COLORS, alabels, afracs, APEX_COLORS, "% of the read pairs assigned to genes")
    axes[0].legend(fontsize=7, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, title="gene class")
    stacked_barh(axes[1], apex_bt[abt_order], abt_order, abt_colors, alabels, afracs, APEX_COLORS, "% of the read pairs assigned to genes")
    axes[1].legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0), title="gene biotype")
    for ax in axes:
        for k in ("control", "fractionation"):
            ax.axhline(np.where(alibs["kind"] == k)[0][0] - 0.5, color="black", lw=0.8)
    fig.tight_layout()
    savefig(fig, "APEX.percent_of_reads_per_class_and_biotype")
    plt.show()
    agrp = alibs[["kind", "fraction"]]
    display(apex_class.join(agrp).groupby(["kind", "fraction"], sort=False).median().round(2))
    display(apex_bt[abt_order].join(agrp).groupby(["kind", "fraction"], sort=False).median().round(2))

### Enrichment of the biotypes against the controls

As for ONT: $p_b = n_b / (n_b + n_{\text{nuclear pc}})$ per library and the log2 odds ratio against the pooled 18 controls; median over the replicates of each construct (controls grouped by the construct of their cells).

In [ ]:
if APEX_READY:
    aref = alibs.index[alibs["kind"] == "control"]
    apex_p, apex_lor = enrichment_vs_protein_coding(apex_counts, aref)
    akeep = apex_p.columns[(apex_p.loc[aref].mean() >= 0.01) | (apex_p.median() >= 0.01)]
    save_tsv(alibs[["kind", "fraction", "replicate"]].join(apex_lor[akeep]).rename_axis("library").reset_index(), "APEX.biotype_log2_odds_ratio_vs_controls.tsv")
    grp_name = np.where(alibs["kind"] == "control", "control (" + alibs["cells"] + " cells)", alibs["kind"] + ": " + alibs["fraction"])
    ag = apex_lor[akeep].groupby(pd.Series(grp_name, index=alibs.index), sort=False).median()
    acols = apex_p.loc[aref, akeep].mean().sort_values(ascending=False).index
    ag = ag[acols]
    alim = np.nanmax(np.abs(ag.values[np.isfinite(ag.values)]))
    sns.set(font_scale=0.8, style="white")
    fig, ax = plt.subplots(figsize=(0.42 * len(acols) + 3, 0.22 * len(ag) + 2))
    sns.heatmap(ag, cmap="RdBu_r", center=0, vmin=-min(alim, 4), vmax=min(alim, 4), annot=True, fmt=".1f", annot_kws={"fontsize": 5.5}, ax=ax,
                cbar_kws={"label": "log2 odds ratio vs controls\n(biotype relative to nuclear protein-coding)"})
    ax.set(xlabel="", ylabel="")
    for t in ax.get_yticklabels():
        t.set_color(APEX_COLORS.get(t.get_text().split(": ")[-1], "black"))
    savefig(fig, "APEX.biotype_log2_odds_ratio_vs_controls")
    plt.show()
    save_tsv(ag.round(3).rename_axis("group").reset_index(), "APEX.biotype_log2_odds_ratio_vs_controls.by_group.tsv")

#### Interpretation: APEX-seq

- **Reads.** 13–93 M read pairs per library and 70–94% uniquely mapped (pairs as counted by STAR). The share assigned to genes is lowest in the nuclear constructs and the nuclear fraction (72–85% of the uniquely mapped pairs, against 88–98% elsewhere), from intronic reads of pre-mRNA.
- **Composition (median over libraries).**
  - Nuclear protein-coding genes take 82–95% of the assigned pairs in all libraries except MITO (38%).
  - The mitochondrially encoded genes take 59% in MITO, 10% in the cytosolic fraction, 1.4–4% elsewhere (controls 3.9%).
  - lncRNAs take 4.5–4.9% in NLS and LMA and 4.1% in the nuclear fraction, against 1.2–1.6% in the cytosolic constructs and the controls.
  - Pseudogenes take 2–6%. Nuclear rRNA is absent (poly(A)+).
- **Enrichment against the controls (log2 odds ratio relative to nuclear protein-coding genes):**
  - NLS, LMA (and less NIK) and the nuclear fraction enrich lncRNAs, snoRNAs, misc_RNAs, TEC and unprocessed pseudogenes by +1 to +2 (2- to 4-fold), and deplete mitochondrial transcripts.
  - MITO enriches all three mitochondrial biotypes (mRNA, Mt_rRNA, Mt_tRNA) by +4.5 to +5.5 (about 32-fold).
  - NES, ERM and OMM deplete mitochondrial transcripts (−1.3 to −1.8) and small nuclear RNAs (snoRNA −1.1 to −1.4).
  - The cytosolic fraction enriches mitochondrial RNA (+1.0 to +1.6) and depletes snoRNAs (−1.2).
- **KDEL and NucPore barely differ from the controls in any biotype** (|log2 OR| ≤ 0.8), consistent with their weak shape differences in `APEX_seq.fracSanity.ipynb`.
- **The controls of the NLS cells contain less mitochondrial RNA than the other controls** (−1.2 to −1.7). The pooled reference averages over such differences between preparations.

## ONT and APEX-seq side by side

Gene-class composition (median over the libraries) of the references (ONT Total siCtrl, APEX-seq controls) and of the matching fractions (ER ↔ ERM / KDEL, Free ↔ NES / cytosolic fraction, membrane sorts ↔ OMM, nuclear fraction / NLS). The ONT shares are shares of molecules (one read per molecule), the APEX-seq shares are shares of read pairs (long RNAs over-represented).

In [ ]:
if APEX_READY:
    o_groups = {"ONT Total siCtrl": ref, "ONT Free (siCtrl)": libs.index[(libs["fraction"] == "Free") & (libs["condition"] == "siCtrl")],
                "ONT ER (siCtrl)": libs.index[(libs["fraction"] == "ER") & (libs["condition"] == "siCtrl")],
                "ONT TIS (siCtrl)": libs.index[(libs["fraction"] == "TIS") & (libs["condition"] == "siCtrl")],
                **{f"ONT {f} (siCtrl)": libs.index[(libs["fraction"] == f) & (libs["condition"] == "siCtrl")] for f in ("Golgi", "PEX", "Endo", "TJ")}}
    a_groups = {"APEX controls": aref, **{f"APEX {f}": alibs.index[(alibs["fraction"] == f) & (alibs["kind"] == "target")]
                                          for f in ("NES", "ERM", "KDEL", "OMM", "NLS")},
                "fractionation Cyto": alibs.index[alibs["fraction"] == "Cyto"], "fractionation Nuc": alibs.index[alibs["fraction"] == "Nuc"]}
    side = pd.concat([pd.DataFrame({k: ont_class.loc[v].median() for k, v in o_groups.items()}).T,
                      pd.DataFrame({k: apex_class.loc[v].median() for k, v in a_groups.items()}).T])
    save_tsv(side.round(3).rename_axis("group").reset_index(), "ONT_vs_APEX.gene_class_composition.tsv")
    sns.set(font_scale=0.85, style="white")
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(side) + 1.5))
    left = np.zeros(len(side))
    for c in CLASSES:
        ax.barh(np.arange(len(side)), side[c], left=left, color=CLASS_COLORS[c], edgecolor="black", lw=0.2, label=c)
        left += side[c].to_numpy()
    ax.set_yticks(np.arange(len(side)), side.index)
    ax.set_ylim(len(side) - 0.5, -0.5)
    ax.axhline(len(o_groups) - 0.5, color="black", lw=1)
    ax.set(xlabel="% of the reads (ONT) / read pairs (APEX-seq) assigned to genes, median over libraries", xlim=(0, 100))
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    fig.tight_layout()
    savefig(fig, "ONT_vs_APEX.gene_class_composition")
    plt.show()

## Summary

- **ONT:** the FACS-sorted membrane fractions are dominated by mitochondrial RNA (35–50% of the molecules in the ER/TIS batch, PEX and TJ). The input-amount control Totals contain 4–7 times less mitochondrial RNA than the HCT116 Total reference. See the ONT interpretation above.
- **APEX-seq:** the constructs change the biotype composition much less than the ONT sorts. Nuclear protein-coding genes take ≥ 82% of the read pairs everywhere except MITO.
  - The expected compartment signatures are clear: nuclear non-coding RNAs in NLS, LMA and the nuclear fraction; mitochondrial transcripts in MITO and depleted in the cytosolic constructs.
  - KDEL and NucPore are hardly distinguishable from the controls.
- **Side by side:** mitochondrial co-sorting is a feature of the ONT membrane sorts, not of the ER-associated APEX constructs. ONT ER is 35% mitochondrial against APEX ERM 1.4%. Comparisons of the two data sets should therefore use within-gene or within-class quantities ($r_g$ of protein-coding genes), not the raw composition.